# Naive discriminator using ClassificationCLIP (untrained classifier head)  
  
### The CLIP Vision Encoder is loaded with pretrained weights.  
### The linear classification head is randomly initialised (untrained) — making this a naive baseline.
  
Outputs  fakeness score in [0, 1] for a given image:  
* Score close to 1.0 \rightarrow likely AI-generated  
* Score close to 0.0 \rightarrow likely real

In [33]:
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, CLIPImageProcessor
from PIL import Image
from torchvision import transforms
import sys
import os

torch.manual_seed(42)

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

model_path = os.path.join(project_root, "src/model/clip_model")
image_path = os.path.join(project_root, "datasets/genimage/imagenet_ai_0424_sdv5/train/ai/833_sdv5_00068.png")

# --- CLIP preprocessing (must match training params) ---
CLIP_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

In [34]:
class ClassificationCLIP(nn.Module):
    """
    CLIP Vision Encoder + Linear classification head.
    Identical architecture to the team's fine-tuned model,
    but here the classifier head is left randomly initialised (untrained).
    """
    def __init__(self, model_path: str):
        super(ClassificationCLIP, self).__init__()

        print("Loading CLIP Vision Encoder...")
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size

        print("Attaching untrained Classification Head...")
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)
        return logits


def load_model(model_path: str = None):
    """
    Loads ClassificationCLIP with a randomly initialised classifier head (D0 baseline).

    Args:
        model_path: Path to the saved CLIP Vision Encoder. Defaults to ./clip_model.

    Returns:
        model, device
    """
    import os
    if model_path is None:
        model_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "clip_model")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ClassificationCLIP(model_path).to(device)
    model.eval()
    return model, device


def get_fakeness_score(image_path: str, model, device) -> float:
    """
    Computes a fakeness score for a single image.

    Args:
        image_path: Path to the image file.
        model: Loaded ClassificationCLIP model.
        device: torch device.

    Returns:
        float in [0, 1] — higher means more likely AI-generated.
    """
    image = Image.open(image_path).convert("RGB")
    tensor = CLIP_TRANSFORM(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(tensor)
        score = torch.sigmoid(logit).item()

    return score

In [35]:
print("Loading D0 (untrained baseline)...")
model, device = load_model(model_path)
score = get_fakeness_score(image_path, model, device)
print(f"Fakeness Score : {score:.4f}  ({'likely AI-generated' if score > 0.5 else 'likely real'})")
print("Note: Score is expected to be near-random as the classifier head is untrained.")

Loading D0 (untrained baseline)...
Loading CLIP Vision Encoder...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3179.97it/s]
CLIPVisionModel LOAD REPORT from: /home/jandy/code/deepfake-it-till-you-make-it/src/model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0..

Attaching untrained Classification Head...
Fakeness Score : 0.7730  (likely AI-generated)
Note: Score is expected to be near-random as the classifier head is untrained.
